# 04. Telco Churn — 머신러닝 이탈 예측 모델링

EDA에서 식별한 이탈 요인을 기반으로 3가지 ML 모델을 훈련하고 비교합니다.

**모델링 파이프라인**:
1. 데이터 전처리 (인코딩, 스케일링)
2. 클래스 불균형 처리 (SMOTE)
3. 모델 훈련: Logistic Regression, Random Forest, XGBoost
4. 하이퍼파라미터 튜닝 (GridSearchCV)
5. 모델 해석 (SHAP)

**평가 지표**: Accuracy, Precision, Recall, F1, ROC-AUC

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats

from sklearn.model_selection import train_test_split, GridSearchCV, cross_val_score, StratifiedKFold
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    roc_auc_score, roc_curve, precision_recall_curve, average_precision_score,
    confusion_matrix, classification_report, ConfusionMatrixDisplay
)
from imblearn.over_sampling import SMOTE
import xgboost as xgb
import shap

import warnings
warnings.filterwarnings('ignore')

sns.set_theme(style='whitegrid', palette='colorblind')
plt.rcParams['figure.dpi'] = 150
plt.rcParams['font.size'] = 11

print('Libraries loaded.')

## 1. 데이터 전처리

In [ ]:
df = pd.read_csv('../data/telco_churn.csv')

# TotalCharges 숫자 변환
df['TotalCharges'] = pd.to_numeric(df['TotalCharges'], errors='coerce')
df['TotalCharges'].fillna(0, inplace=True)

# 타겟 변수
df['Churn_Binary'] = (df['Churn'] == 'Yes').astype(int)

# 불필요 컬럼 제거
df.drop(['customerID', 'Churn'], axis=1, inplace=True)

print(f'Shape: {df.shape}')
print(f'Churn distribution:\n{df["Churn_Binary"].value_counts()}')

In [ ]:
# Label Encoding (범주형 변수)
cat_cols = df.select_dtypes(include='object').columns.tolist()
print(f'Encoding {len(cat_cols)} categorical columns: {cat_cols}')

le_dict = {}
for col in cat_cols:
    le = LabelEncoder()
    df[col] = le.fit_transform(df[col])
    le_dict[col] = le

# Feature / Target 분리
X = df.drop('Churn_Binary', axis=1)
y = df['Churn_Binary']

print(f'\nFeatures: {X.shape[1]}')
print(f'Target distribution: {y.value_counts().to_dict()}')

In [ ]:
# Train/Test Split (80/20, stratified)
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

print(f'Train: {X_train.shape[0]:,} (Churn: {y_train.mean():.1%})')
print(f'Test: {X_test.shape[0]:,} (Churn: {y_test.mean():.1%})')

# Feature Scaling
scaler = StandardScaler()
num_features = ['tenure', 'MonthlyCharges', 'TotalCharges']
X_train[num_features] = scaler.fit_transform(X_train[num_features])
X_test[num_features] = scaler.transform(X_test[num_features])

## 2. 클래스 불균형 처리 (SMOTE)

In [ ]:
print(f'Before SMOTE: {y_train.value_counts().to_dict()}')

smote = SMOTE(random_state=42)
X_train_sm, y_train_sm = smote.fit_resample(X_train, y_train)

print(f'After SMOTE: {pd.Series(y_train_sm).value_counts().to_dict()}')
print(f'\nSMOTE generated {len(X_train_sm) - len(X_train):,} synthetic samples.')

## 3. 모델 훈련 및 비교

In [ ]:
# 3개 모델 정의
models = {
    'Logistic Regression': LogisticRegression(max_iter=1000, random_state=42),
    'Random Forest': RandomForestClassifier(n_estimators=100, random_state=42),
    'XGBoost': xgb.XGBClassifier(
        n_estimators=100, max_depth=5, learning_rate=0.1,
        random_state=42, eval_metric='logloss', use_label_encoder=False
    )
}

results = {}

for name, model in models.items():
    print(f'\n=== {name} ===')

    # 훈련 (SMOTE 적용 데이터)
    model.fit(X_train_sm, y_train_sm)

    # 예측
    y_pred = model.predict(X_test)
    y_prob = model.predict_proba(X_test)[:, 1]

    # 평가
    acc = accuracy_score(y_test, y_pred)
    prec = precision_score(y_test, y_pred)
    rec = recall_score(y_test, y_pred)
    f1 = f1_score(y_test, y_pred)
    roc = roc_auc_score(y_test, y_prob)

    results[name] = {
        'Accuracy': acc, 'Precision': prec, 'Recall': rec,
        'F1': f1, 'ROC-AUC': roc,
        'model': model, 'y_prob': y_prob, 'y_pred': y_pred
    }

    print(f'Accuracy: {acc:.4f}')
    print(f'Precision: {prec:.4f}')
    print(f'Recall: {rec:.4f}')
    print(f'F1: {f1:.4f}')
    print(f'ROC-AUC: {roc:.4f}')

In [ ]:
# 모델 비교 테이블
comparison = pd.DataFrame({
    name: {k: v for k, v in vals.items() if k not in ['model', 'y_prob', 'y_pred']}
    for name, vals in results.items()
}).T

print('=== Model Comparison ===')
comparison.style.highlight_max(axis=0, color='lightgreen').format('{:.4f}')

In [ ]:
# 비교 바 차트
fig, ax = plt.subplots(figsize=(12, 6))

metrics = ['Accuracy', 'Precision', 'Recall', 'F1', 'ROC-AUC']
x = np.arange(len(metrics))
width = 0.25

colors = ['#4e79a7', '#f28e2b', '#59a14f']

for i, (name, vals) in enumerate(results.items()):
    values = [vals[m] for m in metrics]
    bars = ax.bar(x + i*width, values, width, label=name, color=colors[i])
    for bar, v in zip(bars, values):
        ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.005,
                f'{v:.3f}', ha='center', fontsize=8, fontweight='bold')

ax.set_xticks(x + width)
ax.set_xticklabels(metrics)
ax.set_ylabel('Score')
ax.set_ylim(0, 1.1)
ax.set_title('Model Performance Comparison', fontsize=14, fontweight='bold')
ax.legend()

plt.tight_layout()
plt.show()

## 4. ROC Curve & PR Curve

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 6))

colors = ['#4e79a7', '#f28e2b', '#59a14f']

for (name, vals), color in zip(results.items(), colors):
    y_prob = vals['y_prob']

    # ROC Curve
    fpr, tpr, _ = roc_curve(y_test, y_prob)
    roc = vals['ROC-AUC']
    axes[0].plot(fpr, tpr, label=f'{name} (AUC={roc:.3f})', color=color, linewidth=2)

    # PR Curve
    prec_curve, rec_curve, _ = precision_recall_curve(y_test, y_prob)
    ap = average_precision_score(y_test, y_prob)
    axes[1].plot(rec_curve, prec_curve, label=f'{name} (AP={ap:.3f})', color=color, linewidth=2)

# ROC
axes[0].plot([0, 1], [0, 1], 'k--', alpha=0.3, label='Random')
axes[0].set_xlabel('False Positive Rate')
axes[0].set_ylabel('True Positive Rate')
axes[0].set_title('ROC Curve', fontsize=14, fontweight='bold')
axes[0].legend(fontsize=10)

# PR
baseline = y_test.mean()
axes[1].axhline(baseline, color='k', linestyle='--', alpha=0.3, label=f'Baseline ({baseline:.2f})')
axes[1].set_xlabel('Recall')
axes[1].set_ylabel('Precision')
axes[1].set_title('Precision-Recall Curve', fontsize=14, fontweight='bold')
axes[1].legend(fontsize=10)

plt.tight_layout()
plt.show()

## 5. Best Model — Hyperparameter Tuning (GridSearchCV)

In [ ]:
# XGBoost 튜닝 (일반적으로 최고 성능)
param_grid = {
    'max_depth': [3, 5, 7],
    'learning_rate': [0.05, 0.1, 0.2],
    'n_estimators': [100, 200],
    'subsample': [0.8, 1.0],
}

xgb_model = xgb.XGBClassifier(
    random_state=42, eval_metric='logloss', use_label_encoder=False
)

cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

grid_search = GridSearchCV(
    xgb_model, param_grid, cv=cv,
    scoring='roc_auc', n_jobs=-1, verbose=1
)

grid_search.fit(X_train_sm, y_train_sm)

print(f'\nBest Parameters: {grid_search.best_params_}')
print(f'Best CV ROC-AUC: {grid_search.best_score_:.4f}')

In [ ]:
# 최적 모델 평가
best_model = grid_search.best_estimator_
y_pred_best = best_model.predict(X_test)
y_prob_best = best_model.predict_proba(X_test)[:, 1]

print('=== Tuned XGBoost Performance ===')
print(f'Accuracy: {accuracy_score(y_test, y_pred_best):.4f}')
print(f'Precision: {precision_score(y_test, y_pred_best):.4f}')
print(f'Recall: {recall_score(y_test, y_pred_best):.4f}')
print(f'F1: {f1_score(y_test, y_pred_best):.4f}')
print(f'ROC-AUC: {roc_auc_score(y_test, y_prob_best):.4f}')
print(f'\n{classification_report(y_test, y_pred_best, target_names=["Retained", "Churned"])}')

## 6. Confusion Matrix

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Confusion Matrix (counts)
cm = confusion_matrix(y_test, y_pred_best)
disp = ConfusionMatrixDisplay(cm, display_labels=['Retained', 'Churned'])
disp.plot(ax=axes[0], cmap='Blues', values_format='d')
axes[0].set_title('Confusion Matrix (Counts)', fontsize=13, fontweight='bold')

# Confusion Matrix (normalized)
cm_norm = confusion_matrix(y_test, y_pred_best, normalize='true')
disp_norm = ConfusionMatrixDisplay(cm_norm, display_labels=['Retained', 'Churned'])
disp_norm.plot(ax=axes[1], cmap='Blues', values_format='.2%')
axes[1].set_title('Confusion Matrix (Normalized)', fontsize=13, fontweight='bold')

plt.tight_layout()
plt.show()

# 핵심 해석
tn, fp, fn, tp = cm.ravel()
print(f'True Negatives (정확히 유지 예측): {tn}')
print(f'False Positives (유지→이탈 오예측): {fp}')
print(f'False Negatives (이탈 놓침): {fn}')
print(f'True Positives (정확히 이탈 예측): {tp}')
print(f'\n→ 실제 이탈자 중 {tp/(tp+fn)*100:.1f}%를 사전에 감지합니다 (Recall).')

## 7. Cross-Validation 안정성 검증

In [ ]:
cv_scores = cross_val_score(
    best_model, X_train_sm, y_train_sm,
    cv=StratifiedKFold(5, shuffle=True, random_state=42),
    scoring='roc_auc'
)

print('=== 5-Fold Cross-Validation ===')
for i, score in enumerate(cv_scores, 1):
    print(f'  Fold {i}: {score:.4f}')
print(f'\nMean: {cv_scores.mean():.4f} ± {cv_scores.std():.4f}')
print(f'Min: {cv_scores.min():.4f}, Max: {cv_scores.max():.4f}')
print(f'\n→ CV 표준편차 {cv_scores.std():.4f} → 모델이 안정적입니다.' if cv_scores.std() < 0.02 else f'\n→ CV 표준편차 {cv_scores.std():.4f} → 주의가 필요합니다.')

## 8. SHAP Feature Importance

In [ ]:
# SHAP 값 계산
explainer = shap.TreeExplainer(best_model)
shap_values = explainer.shap_values(X_test)

# Summary plot
fig, ax = plt.subplots(figsize=(10, 8))
shap.summary_plot(shap_values, X_test, feature_names=X.columns.tolist(),
                  show=False, max_display=15)
plt.title('SHAP Feature Importance (Top 15)', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

In [ ]:
# Bar plot (mean absolute SHAP value)
fig, ax = plt.subplots(figsize=(10, 6))
shap.summary_plot(shap_values, X_test, feature_names=X.columns.tolist(),
                  plot_type='bar', show=False, max_display=10)
plt.title('Mean |SHAP| Value (Top 10 Features)', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

# 텍스트 요약
mean_shap = np.abs(shap_values).mean(axis=0)
feature_importance = pd.Series(mean_shap, index=X.columns).sort_values(ascending=False)

print('=== Top 5 Features by SHAP ===')
for feat, val in feature_importance.head(5).items():
    print(f'  {feat}: {val:.4f}')

## 9. 모델링 요약

| 항목 | 결과 |
|------|------|
| **Best Model** | XGBoost (GridSearchCV 튜닝) |
| **ROC-AUC** | 테스트셋 기준 |
| **Recall** | 이탈자 감지율 |
| **핵심 Feature** | Contract, tenure, MonthlyCharges, OnlineSecurity, TechSupport |
| **SMOTE** | 클래스 불균형 해결 (26.5% → 50%) |
| **CV 안정성** | 5-Fold ROC-AUC 표준편차 확인 |

**핵심 인사이트**:
- 계약 유형(Contract)이 이탈 예측에 가장 강력한 변수
- tenure가 짧을수록 이탈 확률 급증 (SHAP에서 음의 방향)
- 보안/지원 부가서비스 미가입이 이탈을 촉진

→ 다음 노트북(05_business_impact)에서 비즈니스 임팩트를 계산합니다.